# FSQ Codebook Utilization Probe

Measures FSQ codebook utilization (% of active codes) and perplexity on the held-out Walker test set (`data/walker_trajs_h5/state_t4165_f100_r84_s43.h5`).

In [1]:
import torch
from torch.utils.data import DataLoader
import numpy as np

from rvsablation.jepa import FSQ_V_JEPA
from rvsablation.utils import get_dataset, seed_everything, read_config, fsq_2_index

# import sys
import gc
from pathlib import Path

# parent_dir = str(Path('../..').resolve())
# if parent_dir not in sys.path:
#     sys.path.append(parent_dir)
# from dmc_dataset.dataset import get_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rng = seed_everything(seed=42)

MODEL = "jepa"
RUN_NAME = "fsq_v_jepa_run_1"
DATASET = "walker"

run_conf = read_config(f"../{MODEL}_runs/{RUN_NAME}/confs/run_config.json")

dataset, testset, _, _ = get_dataset(rng=rng, 
                                     root=f"../data/{DATASET}_trajs_h5/state_t4165_f100_r84_s43.h5", 
                                     sample_num_frames=run_conf["sample_num_frames"], 
                                     sample_num_clips=run_conf["sample_num_clips"], 
                                     data_num_traj=4165, 
                                     data_num_frames=run_conf["data_num_frames"],
                                     split_ratio=0.7,
                                     dataset=DATASET)

dataloader = DataLoader(dataset, batch_size=2048, shuffle=True, num_workers=4, pin_memory=True)
testloader = DataLoader(testset, batch_size=2048, shuffle=True, num_workers=4, pin_memory=True)


model_conf = read_config(f"../{MODEL}_runs/{RUN_NAME}/confs/model_config.json")
model = FSQ_V_JEPA(**model_conf)
model = model.to(device)
model.eval()
model.load_state_dict(torch.load(f"../{MODEL}_runs/{RUN_NAME}/weights/model_4.9m.pt"))

levels = model_conf["fsq_levels"]
levels_max = levels // 2
levels_max = levels_max.reshape(-1)
levels_corrected = (levels_max * 2) + 1 # (fsq_dim,)
radii = torch.flip(levels_corrected, dims=(-1,))
radii = torch.cumprod(radii, dim=-1) / radii
radii = torch.flip(radii, dims=(-1,)) # (fsq_dim, )
levels_max = levels_max.to(device)
radii = radii.to(device)

MAX_UTIL = levels_corrected.prod().item()

🌱 Global seed set to 42


## Compute Utilization & Perplexity

In [2]:
num_samples = len(dataloader.dataset)

all_codes = np.zeros((num_samples * 196,), dtype=np.float32)

current_idx = 0

with torch.inference_mode():
    for X, *_ in dataloader:
        X = X.to(device)
        codes = model(X, ctx_idx=None) # no masking
        b = codes.size(0) *  codes.size(1)
        end_idx = current_idx + b

        all_codes[current_idx:end_idx] = fsq_2_index(
            codes.reshape(-1, model_conf["fsq_dim"]), 
            radii[None, :], 
            levels_max[None, :]).cpu().numpy()
        current_idx = end_idx

unique_codes, counts = np.unique(all_codes, axis=0, return_counts=True)
utilization = len(unique_codes) / MAX_UTIL

# perplexity
p = counts / counts.sum()
entropy = -(p * np.log(p)).sum()
perplexity = np.exp(entropy)
del unique_codes
gc.collect()

print(f"Utilization: {utilization*100:.3f}%")
print(f"Perplexity: {perplexity}")

Utilization: 99.348%
Perplexity: 57073.740255635545
